In [12]:
import pandas as pd
from pathlib import Path

ATTR_COLS = [
    "cell_size", "cell_shape", "nucleus_shape", "nuclear_cytoplasmic_ratio",
    "chromatin_density", "cytoplasm_vacuole", "cytoplasm_texture",
    "cytoplasm_colour", "granule_type", "granule_colour", "granularity",
]

# walk up until we find metadata/ so this runs from anywhere
root = Path.cwd()
while not (root / "metadata" / "metadata.csv").exists() and root != root.parent:
    root = root.parent

frames = []
for name in ("attributes.csv", "manual_attributes.csv"):
    p = root / "metadata" / name
    if p.exists():
        frames.append(pd.read_csv(p))
attr = pd.concat(frames, ignore_index=True)

# attributes.csv has no class, so pull hemascope_label from metadata by image_path
meta = pd.read_csv(root / "metadata" / "metadata.csv", usecols=["image_path", "hemascope_label"])
attr = attr.merge(meta, on="image_path", how="left")

print(len(attr), "labeled cells from source(s):", dict(attr["source"].value_counts()))
attr["hemascope_label"].value_counts()

10298 labeled cells from source(s): {'wbcatt': np.int64(10298)}


hemascope_label
Eosinophil              3117
Segmented Neutrophil    1696
Band Neutrophil         1633
Monocyte                1420
Basophil                1218
Lymphocyte              1214
Name: count, dtype: int64

In [13]:
def breakdown(label: str) -> pd.DataFrame:
    """Full per-attribute value split for one class (percent + raw count)."""
    sub = attr if label is None else attr[attr["hemascope_label"] == label]
    n = len(sub)
    rows = []
    for col in ATTR_COLS:
        for value, count in sub[col].value_counts(dropna=False).items():
            rows.append({"attribute": col, "value": value,
                         "pct": round(100 * count / n, 1), "count": count})
    print(f"{label}: {n} labeled cells")
    return pd.DataFrame(rows)


def summary(label: str) -> pd.DataFrame:
    """One row per attribute: the dominant value and its percent."""
    sub = attr if label is None else attr[attr["hemascope_label"] == label]
    n = len(sub)
    rows = []
    for col in ATTR_COLS:
        vc = sub[col].value_counts(dropna=False)
        rows.append({"attribute": col, "most_common": vc.index[0],
                     "pct": round(100 * vc.iloc[0] / n, 1)})
    print(f"{label}: {n} labeled cells")
    return pd.DataFrame(rows)

In [14]:
breakdown("Segmented Neutrophil")

Segmented Neutrophil: 1696 labeled cells


,attribute,value,pct,count
0,cell_size,big,76.2,1293
1,cell_size,small,23.8,403
2,cell_shape,round,69.9,1185
3,cell_shape,irregular,30.1,511
4,nucleus_shape,unsegmented-band,39.6,671
5,nucleus_shape,segmented-bilobed,35.0,593
6,nucleus_shape,segmented-multilobed,25.0,424
7,nucleus_shape,irregular,0.3,5
8,nucleus_shape,unsegmented-round,0.1,2
9,nucleus_shape,unsegmented-indented,0.1,1


In [15]:
breakdown("Band Neutrophil")

Band Neutrophil: 1633 labeled cells


,attribute,value,pct,count
0,cell_size,small,55.6,908
1,cell_size,big,44.4,725
2,cell_shape,round,85.2,1392
3,cell_shape,irregular,14.8,241
4,nucleus_shape,unsegmented-band,95.2,1555
5,nucleus_shape,segmented-bilobed,2.0,32
6,nucleus_shape,irregular,1.2,20
7,nucleus_shape,segmented-multilobed,1.0,16
8,nucleus_shape,unsegmented-round,0.3,5
9,nucleus_shape,unsegmented-indented,0.3,5


In [16]:
breakdown("Lymphocyte")

Lymphocyte: 1214 labeled cells


,attribute,value,pct,count
0,cell_size,small,99.2,1204
1,cell_size,big,0.8,10
2,cell_shape,round,67.1,814
3,cell_shape,irregular,32.9,400
4,nucleus_shape,unsegmented-round,77.4,940
5,nucleus_shape,unsegmented-indented,18.9,229
6,nucleus_shape,irregular,3.7,45
7,nuclear_cytoplasmic_ratio,high,92.8,1127
8,nuclear_cytoplasmic_ratio,low,7.2,87
9,chromatin_density,densely,97.7,1186


In [17]:
breakdown("Monocyte")

Monocyte: 1420 labeled cells


,attribute,value,pct,count
0,cell_size,big,86.5,1228
1,cell_size,small,13.5,192
2,cell_shape,irregular,55.5,788
3,cell_shape,round,44.5,632
4,nucleus_shape,unsegmented-indented,74.7,1061
5,nucleus_shape,irregular,19.0,270
6,nucleus_shape,unsegmented-round,6.0,85
7,nucleus_shape,segmented-bilobed,0.3,4
8,nuclear_cytoplasmic_ratio,low,92.7,1317
9,nuclear_cytoplasmic_ratio,high,7.3,103


In [18]:
breakdown("Eosinophil")

Eosinophil: 3117 labeled cells


,attribute,value,pct,count
0,cell_size,big,63.3,1973
1,cell_size,small,36.7,1144
2,cell_shape,round,91.7,2859
3,cell_shape,irregular,8.3,258
4,nucleus_shape,segmented-bilobed,68.7,2142
5,nucleus_shape,segmented-multilobed,19.6,611
6,nucleus_shape,unsegmented-band,9.9,310
7,nucleus_shape,irregular,0.8,24
8,nucleus_shape,unsegmented-indented,0.7,21
9,nucleus_shape,unsegmented-round,0.3,9


In [19]:
breakdown("Basophil")

Basophil: 1218 labeled cells


,attribute,value,pct,count
0,cell_size,small,72.7,886
1,cell_size,big,27.3,332
2,cell_shape,round,89.0,1084
3,cell_shape,irregular,11.0,134
4,nucleus_shape,irregular,41.5,506
5,nucleus_shape,segmented-bilobed,28.6,348
6,nucleus_shape,segmented-multilobed,19.1,233
7,nucleus_shape,unsegmented-band,6.4,78
8,nucleus_shape,unsegmented-round,2.3,28
9,nucleus_shape,unsegmented-indented,2.1,25


In [ ]:
summary(None)

None: 10298 labeled cells


,attribute,most_common,pct
0,cell_size,big,54.0
1,cell_shape,round,77.4
2,nucleus_shape,segmented-bilobed,30.3
3,nuclear_cytoplasmic_ratio,low,88.0
4,chromatin_density,densely,91.0
5,cytoplasm_vacuole,no,92.3
6,cytoplasm_texture,clear,80.2
7,cytoplasm_colour,light blue,75.8
8,granule_type,small,32.4
9,granule_colour,pink,31.5
